# Data Preprocessing:

### Load the dataset into a suitable data structure (e.g., pandas DataFrame).
### Handle missing values, if any.
### Explore the dataset to understand its structure and attributes.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df=pd.read_csv("anime.csv")

In [3]:
df.head()

,anime_id,name,genre,type,episodes,rating,members
0,32281,Kimi no Na wa.,"Drama, Romance, School, Supernatural",Movie,1,9.37,200630
1,5114,Fullmetal Alchemist: Brotherhood,"Action, Adventure, Drama, Fantasy, Magic, Mili...",TV,64,9.26,793665
2,28977,Gintama°,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.25,114262
3,9253,Steins;Gate,"Sci-Fi, Thriller",TV,24,9.17,673572
4,9969,Gintama&#039;,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.16,151266


In [4]:
df.shape

(12294, 7)

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12294 entries, 0 to 12293
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   anime_id  12294 non-null  int64  
 1   name      12294 non-null  object 
 2   genre     12232 non-null  object 
 3   type      12269 non-null  object 
 4   episodes  12294 non-null  object 
 5   rating    12064 non-null  float64
 6   members   12294 non-null  int64  
dtypes: float64(1), int64(2), object(4)
memory usage: 672.5+ KB


In [6]:
df.isnull().sum()

anime_id      0
name          0
genre        62
type         25
episodes      0
rating      230
members       0
dtype: int64

In [7]:
df=df.dropna(subset=['genre','type'])

In [9]:
df.isnull().sum() # dropped because small amount of data is missing and also categorical

anime_id      0
name          0
genre         0
type          0
episodes      0
rating      193
members       0
dtype: int64

In [11]:
df['rating'] = df['rating'].fillna(df['rating'].median()) # filling the missing values in rating with median

In [12]:
df=df.reset_index(drop=True)

In [13]:
df.isnull().sum()

anime_id    0
name        0
genre       0
type        0
episodes    0
rating      0
members     0
dtype: int64

Now we didnt see any null values

In [14]:
# Replace non-numeric episodes with 0
df['episodes'] = pd.to_numeric(df['episodes'], errors='coerce').fillna(0).astype(int)

# Feature Extraction:

### Decide on the features that will be used for computing similarity (e.g., genres, user ratings).
### Convert categorical features into numerical representations if necessary.
### Normalize numerical features if required.

## One hot encoding for genre

In [20]:
genres = df['genre'].str.get_dummies(sep=',') # splitting genres and creating dummy variables

#Merging with main dataframe

df=pd.concat([df,genres],axis=1)

# drop original genre columns 
df=df.drop('genre',axis=1)

# normalizing numeric features

In [22]:
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
df[['episodes','rating','members']]=scaler.fit_transform(df[['episodes','rating','members']])

In [24]:
# Numeric features we scaled
numeric_features = ['episodes', 'rating', 'members']

# Genre columns are all columns except original anime info
genre_columns = df.columns.difference(['anime_id', 'name', 'type'] + numeric_features)

# Feature matrix for similarity
X = df[genre_columns.tolist() + numeric_features]

print(X.shape)

(12210, 85)


In [28]:
from sklearn.metrics.pairwise import cosine_similarity

cosine_sim = cosine_similarity(X, X)
print(cosine_sim.shape)  # Should be (num_anime, num_anime)

(12210, 12210)


In [32]:
indices = pd.Series(df.index, index=df['name']).drop_duplicates()
# creates a mapping

In [33]:
print(indices["Naruto"])

841


# Recommendation System:

### Design a function to recommend anime based on cosine similarity.
### Given a target anime, recommend a list of similar anime based on cosine similarity scores.
### Experiment with different threshold values for similarity scores to adjust the recommendation list size.
### Analyze the performance of the recommendation system and identify areas of improvement.


In [36]:
def recommend_anime(title, top_n=5):
    
    # Getting index of the anime
    idx = indices[title]
    
    # Getting similarity scores of that anime with all others
    sim_scores = list(enumerate(cosine_sim[idx]))
    
    # Sort anime based on similarity score (descending)
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    
    # Remove the first one (itself)
    sim_scores = sim_scores[1:top_n+1]
    
    # Get indices of recommended anime
    anime_indices = [i[0] for i in sim_scores]
    
    #  Return anime names
    return df['name'].iloc[anime_indices]

In [37]:
print(recommend_anime("Naruto", 5))
print(recommend_anime("Death Note", 5))
print(recommend_anime("Kimi no Na wa.", 5))

615                                    Naruto: Shippuuden
1472          Naruto: Shippuuden Movie 4 - The Lost Tower
1573    Naruto: Shippuuden Movie 3 - Hi no Ishi wo Tsu...
486                              Boruto: Naruto the Movie
1343                                          Naruto x UT
Name: name, dtype: object
778                    Death Note Rewrite
144         Higurashi no Naku Koro ni Kai
833              Jigoku Shoujo Mitsuganae
2691    Yakushiji Ryouko no Kaiki Jikenbo
6320              Saint Luminous Jogakuin
Name: name, dtype: object
5803                          Wind: A Breath of Heart OVA
6391                         Wind: A Breath of Heart (TV)
504     Clannad: After Story - Mou Hitotsu no Sekai, K...
208                         Kokoro ga Sakebitagatterunda.
1201                       Angel Beats!: Another Epilogue
Name: name, dtype: object


The recommendation function uses cosine similarity to find anime that are similar to a given title. It converts the anime name into an index, retrieves similarity scores, sorts them in descending order, and selects the top similar anime while excluding the input anime itself. The function then returns the names of the recommended anime.

In [38]:
def recommend_with_threshold(title, threshold=0.6):
    
    #  Get index
    idx = indices[title]
    
    #  Get similarity scores
    sim_scores = list(enumerate(cosine_sim[idx]))
    
    #  Filter based on threshold
    sim_scores = [x for x in sim_scores if x[1] >= threshold and x[0] != idx]
    
    #  Sort descending
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    
    #  Extract indices
    anime_indices = [i[0] for i in sim_scores]
    
    #  Return names
    return df['name'].iloc[anime_indices]

In [39]:
print(recommend_with_threshold("Naruto", 0.8))
print(recommend_with_threshold("Naruto", 0.6))
print(recommend_with_threshold("Naruto", 0.4))

615                                    Naruto: Shippuuden
1472          Naruto: Shippuuden Movie 4 - The Lost Tower
1573    Naruto: Shippuuden Movie 3 - Hi no Ishi wo Tsu...
486                              Boruto: Naruto the Movie
1343                                          Naruto x UT
2996    Naruto Soyokazeden Movie: Naruto to Mashin to ...
1103    Boruto: Naruto the Movie - Naruto ga Hokage ni...
2458                 Naruto Shippuuden: Sunny Side Battle
175                                Katekyo Hitman Reborn!
7617                              Kyutai Panic Adventure!
7819                        Battle Spirits: Ryuuko no Ken
206                                         Dragon Ball Z
588                                       Dragon Ball Kai
582                                                Bleach
1930                                    Dragon Ball Super
2615                                           Medaka Box
3037                                         Tenjou Tenge
1209          

the above output looks messy, for better understanding we write another code

In [40]:
result = recommend_with_threshold("Naruto", 0.4)

print(len(result))       # number of recommendations
print(result.head(10))   # first 10 only

1648
615                                    Naruto: Shippuuden
1472          Naruto: Shippuuden Movie 4 - The Lost Tower
1573    Naruto: Shippuuden Movie 3 - Hi no Ishi wo Tsu...
486                              Boruto: Naruto the Movie
1343                                          Naruto x UT
2996    Naruto Soyokazeden Movie: Naruto to Mashin to ...
1103    Boruto: Naruto the Movie - Naruto ga Hokage ni...
2458                 Naruto Shippuuden: Sunny Side Battle
175                                Katekyo Hitman Reborn!
7617                              Kyutai Panic Adventure!
Name: name, dtype: object


In [41]:
# comparing the thresholds 
for t in [0.8, 0.6, 0.4]:
    result = recommend_with_threshold("Naruto", t)
    print(f"\nThreshold: {t}")
    print("Count:", len(result))
    print(result.head(5))


Threshold: 0.8
Count: 26
615                                    Naruto: Shippuuden
1472          Naruto: Shippuuden Movie 4 - The Lost Tower
1573    Naruto: Shippuuden Movie 3 - Hi no Ishi wo Tsu...
486                              Boruto: Naruto the Movie
1343                                          Naruto x UT
Name: name, dtype: object

Threshold: 0.6
Count: 415
615                                    Naruto: Shippuuden
1472          Naruto: Shippuuden Movie 4 - The Lost Tower
1573    Naruto: Shippuuden Movie 3 - Hi no Ishi wo Tsu...
486                              Boruto: Naruto the Movie
1343                                          Naruto x UT
Name: name, dtype: object

Threshold: 0.4
Count: 1648
615                                    Naruto: Shippuuden
1472          Naruto: Shippuuden Movie 4 - The Lost Tower
1573    Naruto: Shippuuden Movie 3 - Hi no Ishi wo Tsu...
486                              Boruto: Naruto the Movie
1343                                          Naruto x 

In addition to recommending a fixed number of anime, a threshold-based approach is used to filter recommendations based on similarity scores.

In this method, only those anime whose cosine similarity score is greater than or equal to a specified threshold are recommended. This helps in controlling the quality of recommendations.

A higher threshold (e.g., 0.8) returns fewer but highly similar anime, ensuring better accuracy. A lower threshold (e.g., 0.4) returns a larger number of anime, but some of them may not be very relevant. A moderate threshold (e.g., 0.6) provides a balance between accuracy and the number of recommendations.

Thus, the threshold value plays an important role in adjusting the recommendation list size and quality.

Performance Analysis

The recommendation system uses cosine similarity to suggest anime based on genre and numerical features such as rating, number of episodes, and members.

The system performs well in identifying similar anime. For example, when a popular anime like Naruto is given as input, the system recommends related series and movies from the same franchise, showing high similarity. For other anime like Death Note, the system suggests anime with similar themes such as psychological and thriller genres, indicating that the model effectively captures content-based similarity.

The threshold-based approach shows that the performance of the system depends on the similarity threshold value. A higher threshold (e.g., 0.8) results in fewer but more accurate recommendations, while a lower threshold (e.g., 0.4) produces a large number of recommendations with reduced relevance. A moderate threshold (e.g., 0.6) provides a balance between accuracy and diversity.

Areas of Improvement

Despite its effectiveness, the recommendation system has some limitations:

It relies only on genre and basic numerical features, which may not fully represent the content of anime.
It does not consider user preferences or viewing history, so recommendations are not personalized.
It may produce less meaningful recommendations for less popular or niche anime.
All features are treated equally, without assigning importance to more significant features like rating.
Conclusion

Overall, the recommendation system performs well for content-based filtering and provides meaningful suggestions. However, its performance can be improved by incorporating user-based data and more advanced feature representations.

# interview questions
### 1. Difference between User-Based and Item-Based Collaborative Filtering

User-based collaborative filtering recommends items based on similar users. If two users have similar interests, the system suggests items liked by one user to the other.

Item-based collaborative filtering recommends items based on similarity between items. If a user likes one item, the system suggests other similar items.

## 2. What is Collaborative Filtering and how does it work

Collaborative filtering is a recommendation method that uses user behavior, such as ratings or preferences, to suggest items.

It works by finding similarities between users or items and recommending items based on those similarities. It does not depend on item features but uses user interaction data.